# Event Planning Multi-Agent System

Four **single-purpose** agents collaborate on planning a live event. Each agent:

1. Has its own **focused system prompt** (no overlap in responsibilities).
2. Declares a **Pydantic input/output schema**, which doubles as a machine-readable **JSON Schema** (`model_json_schema()`).
3. Returns **structured JSON** validated against its output schema.
4. Depends on an injected, duck-typed `LLMClient`, so it can be unit-tested **in isolation** with a deterministic fake backend - no network access or API credentials required.

| # | Agent | Responsibility |
|---|-------|-----------------|
| 1 | `VenueAgent` | Recommends venues from headcount, budget, location, event type |
| 2 | `CateringLogisticsAgent` | Proposes menus, service style, and estimated catering costs |
| 3 | `AVVendorCoordinationAgent` | Suggests AV setups and third-party vendors |
| 4 | `AgendaAssemblyAgent` | Assembles a run-of-show from the other three agents' outputs |

**Deliverables in this notebook:**
- Four agent classes with independent system prompts (Sections 3-6)
- JSON Schema definitions for each agent's input/output (Section 7)
- Unit tests proving each agent runs correctly in isolation (Section 8)


## 1. Imports and shared event vocabulary

In [ ]:
import json
import os
import sys
import unittest
from enum import Enum
from typing import Any, Dict, List, Literal, Optional, Protocol, Type

from pydantic import BaseModel, Field, ValidationError

# Make the repo-root `init_creds.py` importable - only needed if/when the real
# Azure OpenAI-backed client below is actually instantiated.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)


class EventType(str, Enum):
    """Shared event taxonomy used as input by every agent."""

    CONFERENCE = "conference"
    WEDDING = "wedding"
    CORPORATE_OFFSITE = "corporate_offsite"
    PRODUCT_LAUNCH = "product_launch"
    GALA = "gala"
    WORKSHOP = "workshop"
    OTHER = "other"


## 2. LLM client abstraction

Every agent depends on a minimal `LLMClient` **protocol** rather than a concrete SDK. This is what
makes "runs correctly in isolation" possible: production code can plug in `AzureOpenAIClient`
(using this team's standard Azure OpenAI + `init_creds.py` pattern), while unit tests plug in a
`FakeLLMClient` that returns a canned, schema-valid response with no network access.


In [ ]:
class LLMClient(Protocol):
    """Minimal interface every agent depends on."""

    def generate(
        self,
        system_prompt: str,
        user_payload: Dict[str, Any],
        schema_name: str,
        json_schema: Dict[str, Any],
    ) -> Dict[str, Any]:
        """Return a dict that should validate against `json_schema`."""
        ...


class AzureOpenAIClient:
    """Production LLM backend: Azure OpenAI chat completions with structured JSON output.

    Credentials are resolved lazily (via the repo's `init_creds.py` helpers) only when an
    instance is actually constructed, so importing/using this module never requires network
    access or secrets - that is reserved for `FakeLLMClient` in tests.
    """

    def __init__(self, model: str = "gpt-4o-mini", api_version: str = "2025-04-01-preview"):
        from openai import AzureOpenAI
        import init_creds as creds

        self._client = AzureOpenAI(
            api_key=creds.get_api_key(),
            azure_endpoint=creds.get_endpoint(),
            api_version=api_version,
        )
        self._model = model

    def generate(self, system_prompt, user_payload, schema_name, json_schema):
        response = self._client.chat.completions.create(
            model=self._model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": json.dumps(user_payload)},
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": schema_name,
                    "schema": json_schema,
                    "strict": False,
                },
            },
        )
        return json.loads(response.choices[0].message.content)


class FakeLLMClient:
    """Deterministic stand-in for a real LLM backend, used for isolated unit tests.

    Returns a pre-canned response and records the last request so tests can assert the
    agent sent the expected system prompt / schema to the backend.
    """

    def __init__(self, canned_response: Dict[str, Any]):
        self._canned_response = canned_response
        self.last_call: Optional[Dict[str, Any]] = None

    def generate(self, system_prompt, user_payload, schema_name, json_schema):
        self.last_call = {
            "system_prompt": system_prompt,
            "user_payload": user_payload,
            "schema_name": schema_name,
            "json_schema": json_schema,
        }
        return self._canned_response


## 3. Shared agent scaffolding

`BaseAgent` only wires up validation and the call to `LLMClient.generate`. Each concrete agent
still defines its **own** `system_prompt`, `input_model`, and `output_model` - there is no shared
prompt text between agents.


In [ ]:
class BaseAgent:
    """Shared scaffolding for single-purpose, structured-output agents.

    Subclasses must set `name`, `system_prompt`, `input_model`, and `output_model`.
    """

    name: str
    system_prompt: str
    input_model: Type[BaseModel]
    output_model: Type[BaseModel]

    def __init__(self, llm: LLMClient):
        self._llm = llm

    def run(self, request):
        """Validate `request` against `input_model`, call the LLM, and validate the result
        against `output_model`.

        `request` may be an already-constructed `input_model` instance or a raw dict - both are
        validated the same way, so callers cannot bypass the input schema.
        """
        if not isinstance(request, self.input_model):
            request = self.input_model.model_validate(request)
        payload = request.model_dump(mode="json")
        raw_output = self._llm.generate(
            system_prompt=self.system_prompt,
            user_payload=payload,
            schema_name=self.name,
            json_schema=self.output_model.model_json_schema(),
        )
        return self.output_model.model_validate(raw_output)

    @classmethod
    def input_json_schema(cls) -> Dict[str, Any]:
        return cls.input_model.model_json_schema()

    @classmethod
    def output_json_schema(cls) -> Dict[str, Any]:
        return cls.output_model.model_json_schema()


## 4. Venue Agent

Recommends candidate venues from **headcount, budget, location, and event type** only. It never
proposes menus, AV equipment, or scheduling.


In [ ]:
class VenueRequest(BaseModel):
    event_type: EventType
    headcount: int = Field(gt=0, description="Expected number of attendees")
    budget_usd: float = Field(gt=0, description="Total budget available for the venue")
    location: str = Field(min_length=1, description="City or region where the venue is needed")
    event_date: Optional[str] = Field(default=None, description="ISO date (YYYY-MM-DD) if known")
    special_requirements: List[str] = Field(
        default_factory=list,
        description="e.g. 'wheelchair accessible', 'outdoor space', 'on-site parking'",
    )


class VenueOption(BaseModel):
    name: str
    area: str = Field(description="Neighborhood or district within the requested location")
    capacity: int = Field(gt=0)
    estimated_cost_usd: float = Field(ge=0)
    amenities: List[str] = Field(default_factory=list)
    rationale: str = Field(description="Why this venue fits the brief")


class VenueRecommendation(BaseModel):
    recommended_venues: List[VenueOption] = Field(min_length=1)
    assumptions: List[str] = Field(default_factory=list)
    notes: Optional[str] = None


class VenueAgent(BaseAgent):
    name = "venue_recommendation_agent"
    input_model = VenueRequest
    output_model = VenueRecommendation
    system_prompt = (
        "You are a single-purpose venue recommendation specialist for live events.\n"
        "Given an event's headcount, budget, location, and event type, propose 2-4 realistic "
        "candidate venues that fit the constraints.\n"
        "Rules:\n"
        "- Never recommend a venue with capacity below the requested headcount.\n"
        "- Keep the total estimated venue cost within budget where possible; if it is not "
        "possible, say so explicitly in `assumptions` rather than silently exceeding budget.\n"
        "- Only reason about venues and their suitability. Do not propose menus, AV equipment, "
        "vendors, or schedules - those are handled by other agents.\n"
        "- Respond ONLY with the structured JSON object defined by the provided schema."
    )


## 5. Catering / Logistics Agent

Proposes a **menu, service style, staffing notes, and cost estimate** only. It never selects a
venue or AV equipment.


In [ ]:
class CateringRequest(BaseModel):
    event_type: EventType
    headcount: int = Field(gt=0)
    budget_usd: float = Field(gt=0)
    meal_period: Literal["breakfast", "brunch", "lunch", "dinner", "reception"]
    dietary_restrictions: List[str] = Field(
        default_factory=list, description="e.g. 'vegetarian', 'gluten-free', 'halal'"
    )
    service_style_preference: Optional[
        Literal["plated", "buffet", "family_style", "food_stations", "cocktail"]
    ] = None


class MenuItem(BaseModel):
    course: str = Field(description="e.g. 'appetizer', 'main', 'dessert', 'beverage'")
    description: str
    dietary_tags: List[str] = Field(default_factory=list)


class CateringPlan(BaseModel):
    service_style: str
    menu_items: List[MenuItem] = Field(min_length=1)
    staffing_notes: str
    estimated_cost_usd: float = Field(ge=0)
    cost_per_head_usd: float = Field(ge=0)
    assumptions: List[str] = Field(default_factory=list)


class CateringLogisticsAgent(BaseAgent):
    name = "catering_logistics_agent"
    input_model = CateringRequest
    output_model = CateringPlan
    system_prompt = (
        "You are a single-purpose catering and food-service logistics specialist.\n"
        "Given an event's type, headcount, budget, meal period, and dietary restrictions, "
        "propose ONE coherent catering plan.\n"
        "Rules:\n"
        "- Every dietary restriction supplied by the requester must be covered by at least one "
        "menu item's `dietary_tags`.\n"
        "- Pick a `service_style` appropriate for the headcount and budget (e.g. buffet for large "
        "budget-conscious groups, plated for formal/smaller events).\n"
        "- `estimated_cost_usd` must approximately equal `cost_per_head_usd` times the headcount.\n"
        "- Only reason about food, beverage, and catering staffing. Do not propose venues, AV "
        "equipment, vendors, or schedules - those are handled by other agents.\n"
        "- Respond ONLY with the structured JSON object defined by the provided schema."
    )


## 6. AV / Vendor-Coordination Agent

Suggests **audiovisual setups and third-party vendors** only. It never proposes a venue or a
menu.


In [ ]:
class AVRequest(BaseModel):
    event_type: EventType
    headcount: int = Field(gt=0)
    budget_usd: float = Field(gt=0)
    venue_type: Optional[str] = Field(
        default=None, description="e.g. 'ballroom', 'outdoor', 'theater' - from the venue agent's output"
    )
    special_requirements: List[str] = Field(
        default_factory=list, description="e.g. 'livestream', 'hybrid attendance', 'stage lighting'"
    )


class AVSetupItem(BaseModel):
    category: str = Field(description="e.g. 'audio', 'video', 'lighting', 'staging', 'connectivity'")
    description: str
    quantity: int = Field(ge=1)


class VendorSuggestion(BaseModel):
    vendor_role: str = Field(description="e.g. 'AV production company', 'livestream provider'")
    notes: str


class AVPlan(BaseModel):
    av_setup: List[AVSetupItem] = Field(min_length=1)
    recommended_vendors: List[VendorSuggestion] = Field(min_length=1)
    estimated_cost_usd: float = Field(ge=0)
    setup_lead_time_hours: float = Field(ge=0)
    assumptions: List[str] = Field(default_factory=list)


class AVVendorCoordinationAgent(BaseAgent):
    name = "av_vendor_coordination_agent"
    input_model = AVRequest
    output_model = AVPlan
    system_prompt = (
        "You are a single-purpose audiovisual and third-party vendor coordination specialist.\n"
        "Given an event's type, headcount, budget, venue type, and special requirements, propose "
        "ONE AV setup plan and the third-party vendors needed to deliver it.\n"
        "Rules:\n"
        "- Cover every item in `special_requirements` with at least one `av_setup` entry or "
        "`recommended_vendors` entry (e.g. 'livestream' must map to a livestream provider).\n"
        "- Scale quantities (microphones, screens, speakers) to the headcount.\n"
        "- Only reason about audiovisual equipment and vendor coordination. Do not propose "
        "venues, menus, or schedules - those are handled by other agents.\n"
        "- Respond ONLY with the structured JSON object defined by the provided schema."
    )


## 7. Agenda-Assembly Agent

Takes the **outputs of the other three agents** (plus basic timing) and assembles a single
run-of-show. It never invents its own venue, menu, or AV decisions.


In [ ]:
class AgendaRequest(BaseModel):
    event_name: str
    event_date: str = Field(description="ISO date (YYYY-MM-DD)")
    start_time: str = Field(description="24h HH:MM")
    end_time: str = Field(description="24h HH:MM")
    venue: VenueRecommendation = Field(description="Output of VenueAgent")
    catering: CateringPlan = Field(description="Output of CateringLogisticsAgent")
    av: AVPlan = Field(description="Output of AVVendorCoordinationAgent")


class RunOfShowItem(BaseModel):
    start_time: str = Field(description="24h HH:MM")
    end_time: str = Field(description="24h HH:MM")
    activity: str
    owner: str = Field(description="e.g. 'Catering team', 'AV crew', 'Host'")
    location: Optional[str] = None
    notes: Optional[str] = None


class RunOfShow(BaseModel):
    event_name: str
    event_date: str
    selected_venue: str = Field(description="Name of the venue chosen from `venue.recommended_venues`")
    run_of_show: List[RunOfShowItem] = Field(min_length=1)
    overall_notes: List[str] = Field(default_factory=list)


class AgendaAssemblyAgent(BaseAgent):
    name = "agenda_assembly_agent"
    input_model = AgendaRequest
    output_model = RunOfShow
    system_prompt = (
        "You are a single-purpose agenda-assembly specialist.\n"
        "You are given the venue options, catering plan, and AV plan produced by other agents, "
        "plus the event's name, date, and start/end time.\n"
        "Rules:\n"
        "- Choose exactly ONE venue from `venue.recommended_venues` and name it in "
        "`selected_venue`.\n"
        "- Sequence catering service and AV setup/teardown alongside the event's main activities "
        "into a single chronological `run_of_show`, fully contained within start_time/end_time "
        "(AV setup may start before `start_time`).\n"
        "- Do NOT invent new venues, menu items, or AV equipment - only sequence and schedule "
        "what you were given.\n"
        "- Respond ONLY with the structured JSON object defined by the provided schema."
    )


## 8. JSON Schema definitions

Each agent's input and output schema is derived directly from its Pydantic model (the single
source of truth) and exported to `schemas/` next to this notebook for review outside the
notebook.


In [ ]:
AGENTS = [VenueAgent, CateringLogisticsAgent, AVVendorCoordinationAgent, AgendaAssemblyAgent]

SCHEMA_DIR = os.path.join(os.getcwd(), "schemas")
os.makedirs(SCHEMA_DIR, exist_ok=True)

for agent_cls in AGENTS:
    input_schema = agent_cls.input_json_schema()
    output_schema = agent_cls.output_json_schema()

    with open(os.path.join(SCHEMA_DIR, f"{agent_cls.name}.input.schema.json"), "w") as f:
        json.dump(input_schema, f, indent=2)
    with open(os.path.join(SCHEMA_DIR, f"{agent_cls.name}.output.schema.json"), "w") as f:
        json.dump(output_schema, f, indent=2)

    print(f"=== {agent_cls.name} ===")
    print(f"  input  -> {agent_cls.input_model.__name__}  "
          f"required={input_schema.get('required', [])}")
    print(f"  output -> {agent_cls.output_model.__name__}  "
          f"required={output_schema.get('required', [])}")

print("\nSchemas written to:", SCHEMA_DIR)


## 9. Unit tests - each agent runs correctly in isolation

Every test backs its agent with a `FakeLLMClient`, so these tests need **no network access and
no API credentials**, and prove:
- the agent validates its input schema,
- it forwards the correct system prompt / output schema to the LLM backend, and
- it validates the backend's response against its output schema before returning it.


In [ ]:
class VenueAgentTests(unittest.TestCase):
    def test_runs_in_isolation_and_returns_valid_output(self):
        canned = {
            "recommended_venues": [
                {
                    "name": "Lakeside Conference Center",
                    "area": "Downtown Austin",
                    "capacity": 180,
                    "estimated_cost_usd": 9500.0,
                    "amenities": ["wifi", "AV-ready stage", "on-site parking"],
                    "rationale": "Capacity and cost comfortably fit the brief.",
                }
            ],
            "assumptions": ["Pricing assumes a weekday booking."],
            "notes": None,
        }
        fake_llm = FakeLLMClient(canned)
        agent = VenueAgent(llm=fake_llm)
        request = VenueRequest(
            event_type=EventType.CONFERENCE,
            headcount=150,
            budget_usd=12000,
            location="Austin, TX",
            event_date="2026-03-10",
            special_requirements=["wheelchair accessible"],
        )

        result = agent.run(request)

        self.assertIsInstance(result, VenueRecommendation)
        self.assertEqual(len(result.recommended_venues), 1)
        self.assertGreaterEqual(result.recommended_venues[0].capacity, request.headcount)
        self.assertEqual(fake_llm.last_call["schema_name"], "venue_recommendation_agent")
        self.assertIn("venue", fake_llm.last_call["system_prompt"].lower())
        self.assertEqual(fake_llm.last_call["user_payload"]["location"], "Austin, TX")

    def test_rejects_invalid_input(self):
        agent = VenueAgent(llm=FakeLLMClient({}))
        with self.assertRaises(ValidationError):
            agent.run(
                {
                    "event_type": "conference",
                    "headcount": -5,  # must be > 0
                    "budget_usd": 100,
                    "location": "NYC",
                }
            )


class CateringLogisticsAgentTests(unittest.TestCase):
    def test_runs_in_isolation_and_returns_valid_output(self):
        canned = {
            "service_style": "buffet",
            "menu_items": [
                {"course": "main", "description": "Herb-roasted chicken", "dietary_tags": []},
                {
                    "course": "main",
                    "description": "Grilled vegetable stack",
                    "dietary_tags": ["vegetarian", "gluten-free"],
                },
            ],
            "staffing_notes": "2 chefs, 4 servers for 150 guests.",
            "estimated_cost_usd": 7500.0,
            "cost_per_head_usd": 50.0,
            "assumptions": [],
        }
        fake_llm = FakeLLMClient(canned)
        agent = CateringLogisticsAgent(llm=fake_llm)
        request = CateringRequest(
            event_type=EventType.CONFERENCE,
            headcount=150,
            budget_usd=8000,
            meal_period="lunch",
            dietary_restrictions=["vegetarian", "gluten-free"],
        )

        result = agent.run(request)

        self.assertIsInstance(result, CateringPlan)
        self.assertEqual(result.service_style, "buffet")
        covered_tags = {tag for item in result.menu_items for tag in item.dietary_tags}
        for restriction in request.dietary_restrictions:
            self.assertIn(restriction, covered_tags)
        self.assertEqual(fake_llm.last_call["schema_name"], "catering_logistics_agent")
        self.assertIn("catering", fake_llm.last_call["system_prompt"].lower())

    def test_rejects_invalid_input(self):
        agent = CateringLogisticsAgent(llm=FakeLLMClient({}))
        with self.assertRaises(ValidationError):
            agent.run(
                {
                    "event_type": "conference",
                    "headcount": 100,
                    "budget_usd": 5000,
                    "meal_period": "midnight_snack",  # not a valid Literal option
                }
            )


class AVVendorCoordinationAgentTests(unittest.TestCase):
    def test_runs_in_isolation_and_returns_valid_output(self):
        canned = {
            "av_setup": [
                {"category": "audio", "description": "Wireless lavalier microphones", "quantity": 4},
                {"category": "video", "description": "Livestream camera + encoder", "quantity": 1},
            ],
            "recommended_vendors": [
                {"vendor_role": "AV production company", "notes": "Full-service local AV vendor."},
                {"vendor_role": "Livestream provider", "notes": "Handles multi-platform streaming."},
            ],
            "estimated_cost_usd": 4200.0,
            "setup_lead_time_hours": 3.0,
            "assumptions": [],
        }
        fake_llm = FakeLLMClient(canned)
        agent = AVVendorCoordinationAgent(llm=fake_llm)
        request = AVRequest(
            event_type=EventType.CONFERENCE,
            headcount=150,
            budget_usd=5000,
            venue_type="ballroom",
            special_requirements=["livestream"],
        )

        result = agent.run(request)

        self.assertIsInstance(result, AVPlan)
        self.assertTrue(any("stream" in v.vendor_role.lower() for v in result.recommended_vendors))
        self.assertEqual(fake_llm.last_call["schema_name"], "av_vendor_coordination_agent")
        self.assertIn("audiovisual", fake_llm.last_call["system_prompt"].lower())

    def test_rejects_invalid_input(self):
        agent = AVVendorCoordinationAgent(llm=FakeLLMClient({}))
        with self.assertRaises(ValidationError):
            agent.run({"event_type": "conference", "headcount": 0, "budget_usd": 1000})


class AgendaAssemblyAgentTests(unittest.TestCase):
    def test_runs_in_isolation_and_returns_valid_output(self):
        venue_output = VenueRecommendation(
            recommended_venues=[
                VenueOption(
                    name="Lakeside Conference Center",
                    area="Downtown Austin",
                    capacity=180,
                    estimated_cost_usd=9500.0,
                    amenities=["wifi"],
                    rationale="Fits headcount and budget.",
                )
            ],
            assumptions=[],
        )
        catering_output = CateringPlan(
            service_style="buffet",
            menu_items=[MenuItem(course="main", description="Herb-roasted chicken")],
            staffing_notes="2 chefs, 4 servers.",
            estimated_cost_usd=7500.0,
            cost_per_head_usd=50.0,
        )
        av_output = AVPlan(
            av_setup=[AVSetupItem(category="audio", description="Lavalier mics", quantity=4)],
            recommended_vendors=[VendorSuggestion(vendor_role="AV production company", notes="Local vendor.")],
            estimated_cost_usd=4200.0,
            setup_lead_time_hours=3.0,
        )
        canned = {
            "event_name": "Annual Partner Summit",
            "event_date": "2026-03-10",
            "selected_venue": "Lakeside Conference Center",
            "run_of_show": [
                {
                    "start_time": "07:00",
                    "end_time": "09:00",
                    "activity": "AV setup and sound check",
                    "owner": "AV crew",
                    "location": "Main ballroom",
                    "notes": None,
                },
                {
                    "start_time": "09:00",
                    "end_time": "12:00",
                    "activity": "Keynote and breakout sessions",
                    "owner": "Host",
                    "location": "Main ballroom",
                    "notes": None,
                },
                {
                    "start_time": "12:00",
                    "end_time": "13:00",
                    "activity": "Buffet lunch service",
                    "owner": "Catering team",
                    "location": "Main ballroom",
                    "notes": None,
                },
            ],
            "overall_notes": ["Catering and AV timing confirmed against vendor lead times."],
        }
        fake_llm = FakeLLMClient(canned)
        agent = AgendaAssemblyAgent(llm=fake_llm)
        request = AgendaRequest(
            event_name="Annual Partner Summit",
            event_date="2026-03-10",
            start_time="09:00",
            end_time="17:00",
            venue=venue_output,
            catering=catering_output,
            av=av_output,
        )

        result = agent.run(request)

        self.assertIsInstance(result, RunOfShow)
        self.assertEqual(result.selected_venue, "Lakeside Conference Center")
        self.assertGreaterEqual(len(result.run_of_show), 1)
        # confirm the agenda agent actually received the upstream agents' structured outputs
        sent_payload = fake_llm.last_call["user_payload"]
        self.assertEqual(
            sent_payload["venue"]["recommended_venues"][0]["name"], "Lakeside Conference Center"
        )
        self.assertEqual(sent_payload["catering"]["service_style"], "buffet")
        self.assertEqual(fake_llm.last_call["schema_name"], "agenda_assembly_agent")

    def test_rejects_invalid_input(self):
        agent = AgendaAssemblyAgent(llm=FakeLLMClient({}))
        with self.assertRaises(ValidationError):
            agent.run({"event_name": "Missing fields"})


class JSONSchemaDefinitionTests(unittest.TestCase):
    """Validates deliverable #2: every agent exposes a well-formed input/output JSON Schema."""

    def test_every_agent_exposes_input_and_output_json_schema(self):
        for agent_cls in AGENTS:
            with self.subTest(agent=agent_cls.name):
                input_schema = agent_cls.input_json_schema()
                output_schema = agent_cls.output_json_schema()
                for schema in (input_schema, output_schema):
                    self.assertEqual(schema.get("type"), "object")
                    self.assertIn("properties", schema)
                    self.assertTrue(len(schema["properties"]) > 0)


Run every test above and fail the cell (and therefore the notebook) if any assertion fails.

In [ ]:
loader = unittest.TestLoader()
suite = unittest.TestSuite()
for test_case in [
    VenueAgentTests,
    CateringLogisticsAgentTests,
    AVVendorCoordinationAgentTests,
    AgendaAssemblyAgentTests,
    JSONSchemaDefinitionTests,
]:
    suite.addTests(loader.loadTestsFromTestCase(test_case))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), "One or more agent unit tests failed."


## 10. Bonus: chaining all four agents end-to-end

Not required by the deliverables, but demonstrates the intended composition: the
`AgendaAssemblyAgent` consumes the structured outputs of the other three agents. Still fully
isolated via `FakeLLMClient` - no network access required.


In [ ]:
venue_agent = VenueAgent(llm=FakeLLMClient({
    "recommended_venues": [
        {
            "name": "Riverside Pavilion",
            "area": "Riverside District",
            "capacity": 220,
            "estimated_cost_usd": 11000.0,
            "amenities": ["wifi", "built-in stage", "valet parking"],
            "rationale": "Comfortably exceeds headcount while staying within budget.",
        }
    ],
    "assumptions": [],
    "notes": None,
}))
catering_agent = CateringLogisticsAgent(llm=FakeLLMClient({
    "service_style": "food_stations",
    "menu_items": [
        {"course": "main", "description": "Chef-attended pasta station", "dietary_tags": ["vegetarian"]},
        {"course": "main", "description": "Carving station", "dietary_tags": []},
    ],
    "staffing_notes": "3 chefs, 6 servers for 200 guests.",
    "estimated_cost_usd": 9000.0,
    "cost_per_head_usd": 45.0,
    "assumptions": [],
}))
av_agent = AVVendorCoordinationAgent(llm=FakeLLMClient({
    "av_setup": [
        {"category": "audio", "description": "Line array speakers", "quantity": 2},
        {"category": "lighting", "description": "Stage wash lighting", "quantity": 6},
    ],
    "recommended_vendors": [
        {"vendor_role": "AV production company", "notes": "Handles rigging and sound."},
    ],
    "estimated_cost_usd": 5200.0,
    "setup_lead_time_hours": 4.0,
    "assumptions": [],
}))
agenda_agent = AgendaAssemblyAgent(llm=FakeLLMClient({
    "event_name": "Product Launch Night",
    "event_date": "2026-05-01",
    "selected_venue": "Riverside Pavilion",
    "run_of_show": [
        {"start_time": "14:00", "end_time": "18:00", "activity": "AV setup", "owner": "AV crew", "location": "Riverside Pavilion", "notes": None},
        {"start_time": "18:00", "end_time": "19:00", "activity": "Guest arrival and reception", "owner": "Host", "location": "Riverside Pavilion", "notes": None},
        {"start_time": "19:00", "end_time": "20:00", "activity": "Food station service", "owner": "Catering team", "location": "Riverside Pavilion", "notes": None},
    ],
    "overall_notes": ["Run-of-show assembled from venue, catering, and AV agent outputs."],
}))

venue_result = venue_agent.run(VenueRequest(
    event_type=EventType.PRODUCT_LAUNCH, headcount=200, budget_usd=15000, location="Chicago, IL",
))
catering_result = catering_agent.run(CateringRequest(
    event_type=EventType.PRODUCT_LAUNCH, headcount=200, budget_usd=10000, meal_period="reception",
    dietary_restrictions=["vegetarian"],
))
av_result = av_agent.run(AVRequest(
    event_type=EventType.PRODUCT_LAUNCH, headcount=200, budget_usd=6000, venue_type="pavilion",
    special_requirements=["stage lighting"],
))
agenda_result = agenda_agent.run(AgendaRequest(
    event_name="Product Launch Night", event_date="2026-05-01", start_time="18:00", end_time="22:00",
    venue=venue_result, catering=catering_result, av=av_result,
))

print(agenda_result.model_dump_json(indent=2))
